# 35. Pillow Image Editing — `pillow`
**Python Libraries Lab · Phase 2 · Computer Vision**

_Resize, crop, filter, watermark and convert images._

✅ **Runs in Colab** as-is.

Pillow is the friendly image library for everyday jobs. Open a photo, resize it, crop it, rotate it, add text or a watermark, adjust brightness, and save it in another format. If you are making product thumbnails, social media images or cleaning up a folder of photos, Pillow is the tool.

**In this notebook:** You will take a sample photo and turn it into ShopEasy product images. A square crop, a set of thumbnails, an edited version with filters and brightness, a watermarked copy, and finally a batch job that converts a whole folder at once.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q pillow
import importlib.metadata as md
for pkg in ['pillow']:
    print(pkg, md.version(pkg))

## Step 1 · Open an image
We use a sample photo bundled with scikit-image and turn it into a Pillow image.

In [ ]:
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw, ImageFont, ImageOps
from skimage import data
import matplotlib.pyplot as plt

photo = Image.fromarray(data.coffee())
print(photo.size, photo.mode)
photo.save("coffee.jpg", quality=90)
photo

## Step 2 · Crop, resize and thumbnail
`fit` makes a perfect square crop. `thumbnail` shrinks in place and keeps the shape.

In [ ]:
square = ImageOps.fit(photo, (400, 400))
stretched = photo.resize((300, 300))
thumb = photo.copy(); thumb.thumbnail((200, 200))
print("square", square.size, "| stretched", stretched.size, "| thumbnail", thumb.size)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, im, t in zip(axes, [square, stretched, thumb], ["fit (square crop)", "resize (stretched)", "thumbnail (keeps shape)"]):
    ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.show()

## Step 3 · Filters and adjustments
Blur, sharpen, grayscale, brightness and contrast.

In [ ]:
edits = {
    "blur": square.filter(ImageFilter.GaussianBlur(4)),
    "sharpen": square.filter(ImageFilter.SHARPEN),
    "grayscale": ImageOps.grayscale(square),
    "brighter": ImageEnhance.Brightness(square).enhance(1.4),
    "contrast": ImageEnhance.Contrast(square).enhance(1.8),
    "rotated": square.rotate(20, expand=True, fillcolor="white"),
}
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, (name, im) in zip(axes, edits.items()):
    ax.imshow(im, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.show()

## Step 4 · Add a watermark
Draw semi-transparent text on a transparent layer, then merge it.

In [ ]:
base = square.convert("RGBA")
layer = Image.new("RGBA", base.size, (0, 0, 0, 0))
draw = ImageDraw.Draw(layer)
font = ImageFont.load_default(size=36)
draw.rectangle([0, 340, 400, 400], fill=(0, 0, 0, 120))
draw.text((20, 348), "ShopEasy · Rs. 349", font=font, fill=(255, 255, 255, 230))
marked = Image.alpha_composite(base, layer)
marked.convert("RGB").save("coffee_watermarked.jpg")
marked

## Step 5 · Batch: make thumbnails for a folder
The real-world job: convert every image in a folder to a small JPEG.

In [ ]:
from pathlib import Path
src, dst = Path("products"), Path("thumbs")
src.mkdir(exist_ok=True); dst.mkdir(exist_ok=True)
for i, im in enumerate([data.coffee(), data.astronaut(), data.chelsea(), data.rocket()]):
    Image.fromarray(im).save(src / f"product_{i}.png")

for f in sorted(src.glob("*.png")):
    with Image.open(f) as im:
        im = im.convert("RGB")
        im.thumbnail((160, 160))
        im.save(dst / (f.stem + ".jpg"), quality=85)
for f in sorted(dst.iterdir()):
    print(f.name, Image.open(f).size, f.stat().st_size // 1024, "KB")

## ✍️ Your turn
Make a copy of `photo`, shrink it with `thumbnail((100, 100))`, and store its **size** (a tuple) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    t = photo.copy(); t.thumbnail((100, 100))
    return answer == t.size, f"Expected {t.size}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
t = photo.copy(); t.thumbnail((100, 100))
answer = t.size

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Image**
- **Mode (RGB/L/RGBA)**
- **thumbnail**
- **ImageFilter**
- **ImageDraw**
- **Format**

**Next:** 36. scikit-image Analysis (`scikit-image`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/36_scikit_image.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pillow
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.